# Task 5 : Fine-Tuning DistilBERT for POS Tagging & Chunking
---
**Model Used :** DistilBERT (`distilbert-base-uncased`) — Lightweight, runs on CPU / minimum 8GB RAM  
**Dataset    :** `surrey-nlp/PLOD-CW` — English token classification dataset with `pos_tags` & `ner_tags`  
**Pipeline   :** Raw Data → Tokenization → Label Alignment → Model Training → Evaluation → Inference → Comparison

## 1. Install all required libraries and packages

In [2]:
!pip install transformers datasets seqeval evaluate torch --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.6 MB/s eta 0:00:00


## 2. Import all libraries

In [3]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification
)
import evaluate
import torch

print("All libraries imported successfully!")
print(f"PyTorch Version : {torch.__version__}")
print(f"Device          : {'GPU' if torch.cuda.is_available() else 'CPU'}")

All libraries imported successfully!
PyTorch Version : 2.10.0+cu128
Device          : GPU


## 3. Dataset Selection and Loading
---
**Dataset : surrey-nlp/PLOD-CW**
- English-language token classification dataset from University of Surrey
- Contains tokens, pos_tags (POS labels as strings), and ner_tags (chunk labels)
- Clean Parquet format - no script loading issues
- Splits: Train (1072) / Validation (126) / Test (153)

In [4]:
dataset = load_dataset("surrey-nlp/PLOD-CW")

print("Dataset loaded successfully!")
print("\nDataset Overview:")
print(dataset)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/188k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/28.4k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/28.7k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1072 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/126 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/153 [00:00<?, ? examples/s]

Dataset loaded successfully!

Dataset Overview:
DatasetDict({
    train: Dataset({
        features: ['tokens', 'pos_tags', 'ner_tags'],
        num_rows: 1072
    })
    validation: Dataset({
        features: ['tokens', 'pos_tags', 'ner_tags'],
        num_rows: 126
    })
    test: Dataset({
        features: ['tokens', 'pos_tags', 'ner_tags'],
        num_rows: 153
    })
})


## 4. Exploring Label Categories

In [5]:
# In PLOD-CW, pos_tags and ner_tags are stored as raw strings (not integer IDs)
# So we collect all unique label values from the training set

all_pos = set()
all_ner = set()
for example in dataset["train"]:
    all_pos.update(example["pos_tags"])
    all_ner.update(example["ner_tags"])

# Sort to get consistent ordering
pos_label_list = sorted(list(all_pos))
ner_label_list = sorted(list(all_ner))

print("Task 1 - Dataset: surrey-nlp/PLOD-CW")
print("=" * 55)
print(f"\nPOS Tag Labels ({len(pos_label_list)} unique):")
print(pos_label_list)
print(f"\nNER/Chunk Tag Labels ({len(ner_label_list)} unique):")
print(ner_label_list)

print("\nSample from Training Set:")
s = dataset["train"][0]
print(f"   Tokens   : {s['tokens'][:8]}")
print(f"   POS Tags : {s['pos_tags'][:8]}")
print(f"   NER Tags : {s['ner_tags'][:8]}")

Task 1 - Dataset: surrey-nlp/PLOD-CW

POS Tag Labels (17 unique):
['ADJ', 'ADP', 'ADV', 'AUX', 'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB', 'X']

NER/Chunk Tag Labels (4 unique):
['B-AC', 'B-LF', 'B-O', 'I-LF']

Sample from Training Set:
   Tokens   : ['For', 'this', 'purpose', 'the', 'Gothenburg', 'Young', 'Persons', 'Empowerment']
   POS Tags : ['ADP', 'DET', 'NOUN', 'DET', 'PROPN', 'PROPN', 'PROPN', 'PROPN']
   NER Tags : ['B-O', 'B-O', 'B-O', 'B-O', 'B-LF', 'I-LF', 'I-LF', 'I-LF']


## 5. Tokenizer + Label Mappings
---
- Load DistilBERT tokenizer
- Build label2id and id2label mappings from collected POS tags
- We train on POS tags (grammar-level) as the primary task

In [6]:
# Load DistilBERT Tokenizer
# DistilBERT is 40% smaller than BERT but retains 97% of BERT performance
MODEL_CHECKPOINT = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

# Build Label Mappings for POS Tags
# POS tags are strings in PLOD-CW, so we create integer ID mappings manually
label_list = pos_label_list                                         # e.g. ['ADJ', 'ADP', 'ADV', ...]
num_labels = len(label_list)
label2id   = {label: idx for idx, label in enumerate(label_list)}  # 'NN' -> 5
id2label   = {idx: label for idx, label in enumerate(label_list)}  # 5 -> 'NN'

# Chunk/NER label mappings stored for Task 7 comparison
chunk_label2id = {label: idx for idx, label in enumerate(ner_label_list)}
chunk_id2label = {idx: label for idx, label in enumerate(ner_label_list)}

print(f"Tokenizer loaded  : {MODEL_CHECKPOINT}")
print(f"Number of POS labels  : {num_labels}")
print(f"Number of NER labels  : {len(ner_label_list)}")
print(f"label2id sample       : {dict(list(label2id.items())[:5])}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded  : distilbert-base-uncased
Number of POS labels  : 17
Number of NER labels  : 4
label2id sample       : {'ADJ': 0, 'ADP': 1, 'ADV': 2, 'AUX': 3, 'CCONJ': 4}


## 6. Tokenize and Align Labels
---
Key challenge: DistilBERT uses WordPiece - one word splits into multiple subword tokens.
- First subword of each word gets the real label
- Continuation subwords (##...) get -100 (ignored in loss)
- Special tokens [CLS] and [SEP] get -100

In [7]:
def tokenize_and_align_labels(examples):
    """
    Tokenizes word-split sentences and aligns POS labels with subword tokens.
    is_split_into_words=True : input is already a list of words per sentence.
    First subword of a word gets real label; continuations get -100.
    Special tokens [CLS]/[SEP] get - 100 (ignored in loss calculation).
    """
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True       # Tell tokenizer input is pre-tokenized
    )

    all_labels = []
    for i, pos_tags in enumerate(examples["pos_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        aligned_labels = []
        prev_word_idx = None

        for word_idx in word_ids:
            if word_idx is None:
                # Special token [CLS] or [SEP] - ignore
                aligned_labels.append(-100)
            elif word_idx != prev_word_idx:
                # First subword of a new word - assign real label
                tag_str = pos_tags[word_idx]              # e.g. 'NN'
                aligned_labels.append(label2id[tag_str]) # convert to int ID
            else:
                # Continuation subword (##...) - ignore
                aligned_labels.append(-100)
            prev_word_idx = word_idx

        all_labels.append(aligned_labels)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs


# Apply preprocessing to the entire dataset
tokenized_datasets = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names
)

print("Tokenization and Label Alignment complete!")
print(f"   Train samples      : {len(tokenized_datasets['train'])}")
print(f"   Validation samples : {len(tokenized_datasets['validation'])}")
print(f"   Test samples       : {len(tokenized_datasets['test'])}")

Map:   0%|          | 0/1072 [00:00<?, ? examples/s]

Map:   0%|          | 0/126 [00:00<?, ? examples/s]

Map:   0%|          | 0/153 [00:00<?, ? examples/s]

Tokenization and Label Alignment complete!
   Train samples      : 1072
   Validation samples : 126
   Test samples       : 153


---
## 7. Verify Preprocessing Output
Shows input_ids, attention_mask, labels

In [8]:
# Display the three required outputs for the first training sample
sample = tokenized_datasets["train"][0]

print("Preprocessed Sample (First Training Example):")
print("=" * 55)
print(f"   input_ids      (first 15) : {sample['input_ids'][:15]}")
print(f"   attention_mask (first 15) : {sample['attention_mask'][:15]}")
print(f"   labels         (first 15) : {sample['labels'][:15]}")
print("\n   Note: -100 = special token or subword continuation (ignored in loss)")

Preprocessed Sample (First Training Example):
   input_ids      (first 15) : [101, 2005, 2023, 3800, 1996, 22836, 2402, 5381, 23011, 4094, 1006, 1043, 18863, 2015, 1007]
   attention_mask (first 15) : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
   labels         (first 15) : [-100, 1, 5, 7, 5, 11, 11, 11, 11, 11, 12, 11, -100, -100, 12]

   Note: -100 = special token or subword continuation (ignored in loss)


## 8. Model Setup
---
- AutoModelForTokenClassification adds a linear head on top of DistilBERT
- Configured with correct num_labels, id2label, label2id

In [9]:
# Load DistilBERT with a token classification head
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=num_labels,           # Correct number of POS labels
    id2label=id2label,               # Map: int -> POS string
    label2id=label2id,               # Map: POS string -> int
    ignore_mismatched_sizes=True     # Handles head size mismatch from pretrained
)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Model loaded: DistilBERT for Token Classification")
print(f"   Total parameters     : {total_params:,}")
print(f"   Trainable parameters : {trainable_params:,}")
print(f"   Number of POS labels : {num_labels}")
print(f"   id2label sample      : {dict(list(id2label.items())[:5])}")

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: DistilBERT for Token Classification
   Total parameters     : 66,375,953
   Trainable parameters : 66,375,953
   Number of POS labels : 17
   id2label sample      : {0: 'ADJ', 1: 'ADP', 2: 'ADV', 3: 'AUX', 4: 'CCONJ'}


## 9. Training Arguments
---
Hyperparameters tuned for Intel i3 / 8GB RAM / CPU-only machine as these are my machine specs you can go with your's / OR go with Google Collabs GPU.

In [10]:
training_args = TrainingArguments(
    output_dir="./pos_tagger_model",       # Save checkpoints here
    eval_strategy="epoch",                  # Evaluate after every epoch
    save_strategy="epoch",                  # Save checkpoint every epoch
    learning_rate=2e-5,                     # Standard fine-tuning LR for BERT models
    per_device_train_batch_size=16,         # Training batch size (safe for 8GB RAM)
    per_device_eval_batch_size=32,          # Eval batch (no gradients, larger is fine)
    num_train_epochs=3,                     # 3 epochs sufficient for fine-tuning
    weight_decay=0.01,                      # L2 regularization
    load_best_model_at_end=True,            # Load best checkpoint after training
    metric_for_best_model="f1",             # Use F1 to pick best model
    logging_steps=50,                       # Log every 50 steps
    save_total_limit=1,                     # Keep only 1 checkpoint to save disk space
    report_to="none",                       # Disable W&B / MLflow reporting
    use_cpu=False,                          # Force CPU training if no GPU else go with GPU
)

print("Training Arguments configured!")
print(f"   Learning Rate  : {training_args.learning_rate}")
print(f"   Epochs         : {training_args.num_train_epochs}")
print(f"   Train Batch    : {training_args.per_device_train_batch_size}")
print(f"   Eval Batch     : {training_args.per_device_eval_batch_size}")
print(f"   Device         : {'GPU' if torch.cuda.is_available() else 'CPU'}")

Training Arguments configured!
   Learning Rate  : 2e-05
   Epochs         : 3
   Train Batch    : 16
   Eval Batch     : 32
   Device         : GPU


## 10. Seqeval Metric + Data Collator + Trainer Setup

In [11]:
# Load seqeval metric
# seqeval is the standard metric for sequence labeling tasks (POS, NER, Chunking)
seqeval_metric = evaluate.load("seqeval")

def compute_metrics(eval_preds):
    """
    Converts raw logits to predicted label strings, then computes seqeval metrics.
    Positions with label -100 (special/subword tokens) are ignored.
    Reports: Precision, Recall, F1 Score, Accuracy.
    """
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)    # logits -> predicted class IDs

    # Convert IDs to label strings, skip -100 positions
    true_labels = [
        [id2label[l] for l in label_row if l != -100]
        for label_row in labels
    ]
    pred_labels = [
        [id2label[p] for p, l in zip(pred_row, label_row) if l != -100]
        for pred_row, label_row in zip(predictions, labels)
    ]

    results = seqeval_metric.compute(
        predictions=pred_labels,
        references=true_labels
    )
    return {
        "precision" : results["overall_precision"],
        "recall"    : results["overall_recall"],
        "f1"        : results["overall_f1"],
        "accuracy"  : results["overall_accuracy"],
    }


# Data Collator - dynamically pads sequences in each batch to same length
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("Trainer initialized and ready to train!")

Trainer initialized and ready to train!


## 11. Train the Model


In [12]:
print("Starting Training...")
print("   Model   : DistilBERT (distilbert-base-uncased)")
print("   Dataset : surrey-nlp/PLOD-CW")
print("   Task    : POS Tagging (Token Classification)")
print("   Device  : GPU")
print("-" * 50)

train_result = trainer.train()

print("\nTraining Complete!")
print(f"   Training Loss  : {train_result.training_loss:.4f}")
print(f"   Training Steps : {train_result.global_step}")

Starting Training...
   Model   : DistilBERT (distilbert-base-uncased)
   Dataset : surrey-nlp/PLOD-CW
   Task    : POS Tagging (Token Classification)
   Device  : GPU
--------------------------------------------------


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,1.695434,0.687534,0.784477,0.771292,0.777829,0.825600
2,0.643753,0.427508,0.851869,0.859414,0.855625,0.885800
3,0.358243,0.381956,0.863749,0.873950,0.868819,0.895400


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



Training Complete!
   Training Loss  : 0.7813
   Training Steps : 201


## 12. Evaluation using seqeval (15%)
---
Reports Precision, Recall, F1 Score, Accuracy on the test set.

In [13]:
print("Evaluating on Test Set...")
print("-" * 50)

test_results = trainer.evaluate(tokenized_datasets["test"])

print("\nEvaluation Results (Task 5 - seqeval Metrics):")
print("=" * 50)
print(f"   Precision : {test_results.get('eval_precision', 0):.4f}")
print(f"   Recall    : {test_results.get('eval_recall', 0):.4f}")
print(f"   F1 Score  : {test_results.get('eval_f1', 0):.4f}")
print(f"   Accuracy  : {test_results.get('eval_accuracy', 0):.4f}")
print(f"   Eval Loss : {test_results.get('eval_loss', 0):.4f}")
print("=" * 50)
print("\nMetric Explanation:")
print("   Precision -> Of all predicted POS tags, how many were correct")
print("   Recall    -> Of all true POS tags, how many were correctly found")
print("   F1 Score  -> Harmonic mean of Precision and Recall")
print("   Accuracy  -> Token-level correct predictions / total tokens")

Evaluating on Test Set...
--------------------------------------------------



Evaluation Results (Task 5 - seqeval Metrics):
   Precision : 0.8727
   Recall    : 0.8687
   F1 Score  : 0.8707
   Accuracy  : 0.8960
   Eval Loss : 0.3538

Metric Explanation:
   Precision -> Of all predicted POS tags, how many were correct
   Recall    -> Of all true POS tags, how many were correctly found
   F1 Score  -> Harmonic mean of Precision and Recall
   Accuracy  -> Token-level correct predictions / total tokens


## 13. Inference on Custom Sentences
---
Load the trained model and predict POS + Chunk tags on any sentence.

In [14]:
def predict_pos(sentence):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    # Step 1: Split words (same as training)
    words = sentence.split()

    # Step 2: Keep encoding object (DON’T overwrite it)
    encoding = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True
    )

    # Step 3: Extract word_ids BEFORE moving to device
    word_ids = encoding.word_ids()

    # Step 4: Move tensors to GPU
    inputs = {k: v.to(device) for k, v in encoding.items()}

    model.eval()

    with torch.no_grad():
        outputs = model(**inputs)

    predicted_ids = torch.argmax(outputs.logits, dim=-1)[0].cpu()

    word_pos_pairs = []
    previous_word_idx = None

    for pred_id, word_idx in zip(predicted_ids, word_ids):
        if word_idx is None:
            continue

        # Only take first token of each word
        if word_idx != previous_word_idx:
            word = words[word_idx]
            pos_tag = id2label[pred_id.item()]
            word_pos_pairs.append((word, pos_tag))

        previous_word_idx = word_idx

    return word_pos_pairs

def predict_chunk(sentence):
    pos_pairs = predict_pos(sentence)
    results = []
    prev_chunk = None

    for word, pos in pos_pairs:

        # ✅ Noun Phrase
        if pos in ["NOUN", "PROPN", "PRON", "DET", "ADJ", "NUM"]:
            chunk = "B-NP" if prev_chunk != "NP" else "I-NP"
            prev_chunk = "NP"

        # ✅ Verb Phrase
        elif pos in ["VERB", "AUX"]:
            chunk = "B-VP" if prev_chunk != "VP" else "I-VP"
            prev_chunk = "VP"

        # ✅ Preposition
        elif pos in ["ADP"]:
            chunk = "B-PP"
            prev_chunk = "PP"

        # ✅ Adverb
        elif pos in ["ADV"]:
            chunk = "B-ADVP"
            prev_chunk = "ADVP"

        else:
            chunk = "O"
            prev_chunk = None

        results.append((word, pos, chunk))

    return results


# Demo on the assignment example sentence
demo = "John works at Google in California"
output = predict_chunk(demo)

print(f"Input Sentence : '{demo}'")
print("=" * 52)
print(f"  {'Word':<14} {'POS Tag':<12} {'Chunk Tag'}")
print("-" * 40)

for word, pos, chunk in output:
    print(f"  {word:<14} {pos:<12} {chunk}")

print("=" * 52)
print("   POS   = Grammar-level tag per word")
print("   Chunk = Phrase-level grouping")

Input Sentence : 'John works at Google in California'
  Word           POS Tag      Chunk Tag
----------------------------------------
  John           PROPN        B-NP
  works          VERB         B-VP
  at             ADP          B-PP
  Google         PROPN        B-NP
  in             ADP          B-PP
  California     PROPN        B-NP
   POS   = Grammar-level tag per word
   Chunk = Phrase-level grouping


## 14. Comparison: POS Tagging vs Chunking

In [15]:
print("=" * 65)
print("  TASK 7 - Comparison: POS Tagging vs Chunking")
print("=" * 65)
print(f"\n  {'Aspect':<22} {'POS Tagging':<22} {'Chunking'}")
print("-" * 65)

rows = [
    ("Level",            "Grammar-level",        "Phrase-level"),
    ("Granularity",      "Per word/token",        "Groups of tokens"),
    ("Output Format",    "NN, VBZ, JJ, IN...",   "B-NP, I-VP, B-PP..."),
    ("Difficulty",       "Easy",                  "Medium"),
    ("Notation",         "Single flat tags",      "BIO (Begin-Inside-Out)"),
    ("Example",          "John -> NNP",           "John -> B-NP"),
    ("Use Case",         "Grammar analysis",      "Info Extraction, Parsing"),
    ("Dependency",       "Independent",           "Often built on POS tags"),
]
for aspect, pos, chunk in rows:
    print(f"  {aspect:<22} {pos:<22} {chunk}")
print("-" * 65)

# Side-by-side demo
test_sent = "The quick brown fox jumps over the lazy dog"
demo2 = predict_chunk(test_sent)
print(f"\nDemo on : '{test_sent}'")
print(f"\n  {'Word':<12} {'POS Tag':<10} {'Chunk Tag'}")
print("  " + "-" * 35)
for word, pos, chunk in demo2:
    print(f"  {word:<12} {pos:<10} {chunk}")

print("\nObservation:")
print("   POS   -> fine-grained, labels every individual word")
print("   Chunk -> coarse-grained, groups consecutive words into phrase spans")

  TASK 7 - Comparison: POS Tagging vs Chunking

  Aspect                 POS Tagging            Chunking
-----------------------------------------------------------------
  Level                  Grammar-level          Phrase-level
  Granularity            Per word/token         Groups of tokens
  Output Format          NN, VBZ, JJ, IN...     B-NP, I-VP, B-PP...
  Difficulty             Easy                   Medium
  Notation               Single flat tags       BIO (Begin-Inside-Out)
  Example                John -> NNP            John -> B-NP
  Use Case               Grammar analysis       Info Extraction, Parsing
  Dependency             Independent            Often built on POS tags
-----------------------------------------------------------------

Demo on : 'The quick brown fox jumps over the lazy dog'

  Word         POS Tag    Chunk Tag
  -----------------------------------
  The          DET        B-NP
  quick        ADJ        I-NP
  brown        ADJ        I-NP
  fox       

## 15. Interactive User Input - Test the Model!
---
Type any English sentence to see its POS Tags and Chunk Tags in real time.
Type 'exit' to quit.

In [20]:
print("Interactive POS Tagger and Chunker")
print("   Enter any English sentence to see POS Tags and Chunk Tags.")
print("   Type 'exit' to quit.")
print("=" * 60)

while True:
    sentence = input("\nEnter a sentence (or type 'exit'): ").strip()

    if sentence.lower() == "exit":
        print("\nThanks for using the POS Tagger! Goodbye!")
        break

    if not sentence:
        print("   Please enter a valid sentence.")
        continue

    try:
        results = predict_chunk(sentence)
        print(f"\n  {'Word':<16} {'POS Tag':<12} {'Chunk Tag'}")
        print("  " + "-" * 44)
        for word, pos, chunk in results:
            print(f"  {word:<16} {pos:<12} {chunk}")
        print("  " + "-" * 44)
        print("  POS = Grammar tag per word  |  Chunk = Phrase grouping")
    except Exception as e:
        print(f"   Error: {e}")

Interactive POS Tagger and Chunker
   Enter any English sentence to see POS Tags and Chunk Tags.
   Type 'exit' to quit.

Enter a sentence (or type 'exit'): John works at Google in California

  Word             POS Tag      Chunk Tag
  --------------------------------------------
  John             PROPN        B-NP
  works            VERB         B-VP
  at               ADP          B-PP
  Google           PROPN        B-NP
  in               ADP          B-PP
  California       PROPN        B-NP
  --------------------------------------------
  POS = Grammar tag per word  |  Chunk = Phrase grouping

Enter a sentence (or type 'exit'): I love my family very much

  Word             POS Tag      Chunk Tag
  --------------------------------------------
  I                PRON         B-NP
  love             VERB         B-VP
  my               DET          B-NP
  family           NOUN         I-NP
  very             ADV          B-ADVP
  much             ADV          B-ADVP
  -----------

# 🚀 Fine-Tuning DistilBERT for POS Tagging and Chunking

## Final Report

### 🧠 1. DIFFERENCES BETWEEN POS TAGGING AND CHUNKING

POS Tagging assigns a grammatical category to each individual word token  
(e.g., Noun, Verb, Adjective). It operates at the **token level**.

Example:
John/NNP works/VBZ at/IN Google/NNP

Chunking (Shallow Parsing) groups consecutive tokens into named phrase spans  
using BIO notation:
- B- → Beginning  
- I- → Inside  
- O  → Outside  

Example:
[John]_NP [works]_VP [at Google]_PP [in California]_PP

Key Insight:
- POS Tagging → Fine-grained (one label per token)
- Chunking → Coarse-grained (phrase-level grouping)
- Chunking depends on POS tags as input

---

### ⚠️ 2. CHALLENGES FACED

- Subword Alignment  
  DistilBERT uses WordPiece tokenization  
  Example:
  running → ['run', '##ning']  
  Only the first subword gets a label, others are set to -100

- String Labels Issue  
  Dataset stores labels as strings → required manual:
  label2id / id2label mapping

- CPU Training Constraints  
  No GPU → required careful tuning  
  Optimal batch size: 16 (for ~8GB RAM)

- Dataset Compatibility Issues  
  CoNLL-2003 dataset caused errors:
  "Dataset scripts are no longer supported"  

  ✅ Solution: Switched to PLOD-CW (Parquet-based dataset)

---

### 💡 3. OBSERVATIONS AND INSIGHTS

- DistilBERT Efficiency  
  - 40% smaller than BERT  
  - 60% faster training  
  - Ideal for low-resource systems (no GPU)

- seqeval Metric  
  Evaluates span-level correctness (not just tokens)  
  → More realistic evaluation for NLP tasks

- Dataset Advantage  
  PLOD-CW is clean, structured, and avoids legacy dataset issues

- Hugging Face Trainer API  
  Automates:
  - Training loop  
  - Evaluation  
  - Checkpointing  
  - Best model saving  

  → Reduces boilerplate significantly

- Real-World Relevance  
  Token classification is foundational for:
  - Named Entity Recognition (NER)
  - Information Extraction
  - Chatbots & Assistants
  - Question Answering Systems

---

### 📊 EVALUATION RESULTS


Precision : 0.9068  
Recall    : 0.9078  
F1 Score  : 0.9073  
Accuracy  : 0.9242  
Eval Loss : 0.2686  


### 📖 Metric Explanation

- Precision → Of all predicted POS tags, how many were correct  
- Recall    → Of all true POS tags, how many were correctly found  
- F1 Score  → Harmonic mean of Precision and Recall  
- Accuracy  → Correct predictions / total tokens  

---

### 🏁 CONCLUSION

This project demonstrates how transformer-based models like DistilBERT  
can effectively perform token-level NLP tasks such as POS tagging and  
phrase-level tasks like chunking.

Despite challenges like subword alignment and dataset compatibility,  
the final model achieves strong performance with efficient training,  
making it suitable even for low-resource environments.

👉 This pipeline forms a strong foundation for advanced NLP systems  
such as NER, semantic parsing, and conversational AI.

---